# Basic Analysis Notebook

This notebook performs basic analysis for data that failed quality checks.

In [ ]:
import pandas as pd
import numpy as np
import json

In [ ]:
# Get shared data from input parameters (passed from previous notebook's task_values)
# NO DEFAULT VALUES - fail fast if data is missing
try:
    mean_x = float(dbutils.widgets.get("raw_data_mean_x"))
    mean_y = float(dbutils.widgets.get("raw_data_mean_y"))
    data_size = int(dbutils.widgets.get("data_size_actual"))
except Exception as e:
    raise ValueError(f"Missing required data from validation step: {e}")

# Handle category_distribution JSON parameter
category_dist_str = dbutils.widgets.get("category_distribution")
if not category_dist_str or category_dist_str == "":
    raise ValueError("Missing required category_distribution data from validation step")

try:
    category_dist = json.loads(category_dist_str)
except json.JSONDecodeError as e:
    raise ValueError(f"Invalid category_distribution JSON: {e}")

# Validate we got real data, not defaults
if data_size <= 0:
    raise ValueError(f"Invalid data_size: {data_size}. Validation step may have failed.")

if not category_dist:
    raise ValueError("Empty category_distribution. Validation step may have failed.")

print("📊 Using data passed from validation step:")
print(f"  Mean X: {mean_x:.3f}, Mean Y: {mean_y:.3f}")
print(f"  Data size: {data_size}")
print(f"  Category distribution: {category_dist}")

In [ ]:
# Basic statistical analysis
print("🔍 Performing BASIC analysis (due to data quality issues)...")

# Simple metrics using shared data
mean_magnitude = np.sqrt(mean_x**2 + mean_y**2)
category_count = len(category_dist)
total_samples = sum(category_dist.values())
largest_category = max(category_dist.values()) if category_dist else 0
category_balance = largest_category / total_samples if total_samples > 0 else 0

print(f"  Mean Magnitude: {mean_magnitude:.3f}")
print(f"  Category Count: {category_count}")
print(f"  Category Balance: {category_balance:.3f}")

In [ ]:
# Share basic metrics via task_values for downstream notebooks
dbutils.jobs.taskValues.set("mean_magnitude", mean_magnitude)
dbutils.jobs.taskValues.set("category_count", category_count)
dbutils.jobs.taskValues.set("category_balance", category_balance)
dbutils.jobs.taskValues.set("analysis_type", "basic")

print("✅ Shared basic metrics via task_values")

In [ ]:
# Determine recommendation for next step (more conservative)
data_usable = (
    data_size > 10 and
    category_count >= 2 and
    category_balance < 0.9  # Not too imbalanced
)

if data_usable:
    recommendation = "BASIC_MODELING"
    confidence = "LOW"
    print("🔧 Data is usable for basic modeling with caution")
else:
    recommendation = "DATA_COLLECTION"
    confidence = "NONE"
    print("❌ Data quality too poor - recommend collecting new data")

# Return recommendation for workflow decision
basic_results = {
    "analysis_type": "basic",
    "recommendation": recommendation,
    "confidence": confidence,
    "data_quality_score": 0.3,  # Lower score for basic analysis
    "basic_metrics": {
        "mean_magnitude": mean_magnitude,
        "category_balance": category_balance
    },
    "warnings": ["Data failed quality checks", "Results may be unreliable"]
}

print(f"🎯 Basic Analysis Result: {recommendation} ({confidence} confidence)")

# This exit_value will be used by workflow for next step decision
dbutils.notebook.exit(basic_results)